In [ ]:
import pickle

from darts.models import GlobalNaiveSeasonal

from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES
from aare.params import read_params
from aare.paths import DATA_FOLDER

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
params = read_params()
target = "temp_bern"

In [ ]:
model = GlobalNaiveSeasonal(input_chunk_length=24, output_chunk_length=1)

In [ ]:
features = dict(targets=FEATURES["temp_bern"])
feature_set = FeatureSet(**features, split_params=params["split"])

In [ ]:
targets, _, _ = feature_set.get_train()

In [ ]:
model.fit(targets)

In [ ]:
model.save(str(DATA_FOLDER / "models" / "snaive"), clean=True)

In [ ]:
with open(DATA_FOLDER / "models" / "snaive.features", "wb") as file:
    pickle.dump(features, file)

In [ ]:
from darts.models import GlobalNaiveDrift
from darts.models.forecasting.torch_forecasting_model import TorchForecastingModel
from darts.datasets import AirPassengersDataset

passengers = AirPassengersDataset().load()

# model = LinearRegressionModel(lags=1, output_chunk_length=1)  # works, global but not torch
# model = RNNModel(input_chunk_length=1, n_epochs=10)  # works, torch
# model = GlobalNaiveSeasonal(input_chunk_length=input_steps, output_chunk_length=1)  # doesn't work, torch without weights
# model = GlobalNaiveAggregate(input_chunk_length=1, output_chunk_length=1)  # doesn't work, torch without weights
model = GlobalNaiveDrift(input_chunk_length=1, output_chunk_length=1)  # doesn't work, torch without weights

model.fit(passengers)

path = "model"
model.save(path)  # fails here

# Unfortunately, you cannot use GlobalForecastingModel.load for torch models :(
# model = GlobalForecastingModel.load(path)
model = TorchForecastingModel.load(path)

[BUG] Cannot save global baseline models

**Describe the bug**

Trying to save a global baseline model (GlobalNaiveSeasonal, GlobalNaiveAggregate, GlobalNaiveDrift) throws the following error.

> AttributeError: Saving a checkpoint is only possible if a model is attached to the Trainer. Did you call `Trainer.save_checkpoint()` before calling `Trainer.{fit,validate,test,predict}`?

**To Reproduce**

```python
from darts.models import LinearRegressionModel, RNNModel, GlobalNaiveSeasonal, GlobalNaiveAggregate, GlobalNaiveDrift
from darts.models.forecasting.forecasting_model import GlobalForecastingModel
from darts.models.forecasting.torch_forecasting_model import TorchForecastingModel
from darts.datasets import AirPassengersDataset

passengers = AirPassengersDataset().load()

# model = LinearRegressionModel(lags=1, output_chunk_length=1)  # works, global but not torch
# model = RNNModel(input_chunk_length=1, n_epochs=10)  # works, torch
# model = GlobalNaiveSeasonal(input_chunk_length=input_steps, output_chunk_length=1)  # doesn't work, torch without weights
# model = GlobalNaiveAggregate(input_chunk_length=1, output_chunk_length=1)  # doesn't work, torch without weights
model = GlobalNaiveDrift(input_chunk_length=1, output_chunk_length=1)  # doesn't work, torch without weights

model.fit(passengers)

path = "model"
model.save(path)  # <---- fails here

# make sure loading works too.
# Unfortunately, you cannot use GlobalForecastingModel.load for torch models :(
# model = GlobalForecastingModel.load(path)
model = TorchForecastingModel.load(path)
```

**Expected behavior**

The model is saved with all required information so that it can be loaded with `TorchForecastingModel.load` (or even better `GlobalForecastingModel.load`).

**System (please complete the following information):**
 - Python version: 3.12.9
 - darts version: 0.35.0


In [ ]:
# some actual storage experiments in 09_linear-regression